In [ ]:
import os
os.chdir('/content')
print(os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
!pip install -q fair-esm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 9.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ls -la /content/drive/MyDrive/bs6204/

Mounted at /content/drive
total 269073
drwx------ 2 root root     4096 Oct 15 14:19 .
drwx------ 5 root root     4096 Oct 16 09:33 ..
-rw------- 1 root root   127785 Oct 15 13:50 contacts.npz
-rw------- 1 root root     5609 Oct 14 13:20 eval_harness.py
-rw------- 1 root root     3682 Oct 15 14:32 eval_per_assay.csv
-rw------- 1 root root 13672932 Oct 14 13:35 m1_650_preds.csv
-rw------- 1 root root   812142 Oct 16 09:00 m1_nosingles_final.pt
-rw------- 1 root root 13639914 Oct 16 09:00 m1_ns_preds.csv
-rw------- 1 root root 13681632 Oct 14 13:29 m1_preds.csv
-rw------- 1 root root 12953846 Oct 15 14:26 m2_preds.csv
-rw------- 1 root root 57220648 Oct 15 13:50 pairs_3d.csv
-rw------- 1 root root 53544459 Oct  6 12:18 pairs.csv
-rw------- 1 root root 54129065 Oct 15 14:01 pairs_struct.csv
-rw------- 1 root root    10097 Oct  6 12:18 sequences.csv
-rw------- 1 root root      478 Oct 15 14:35 slice_compare.csv
-rw------- 1 root root    66440 Oct 15 14:35 slice_compare.png
-rw------- 1 root

In [ ]:
!mkdir -p /content/work
!cp /content/drive/MyDrive/bs6204/{train_m1.py,eval_harness.py,pairs.csv,sequences.csv,wt_emb_650.pt} /content/work/
os.chdir('/content/work')
!ls -la

total 91828
drwxr-xr-x 2 root root     4096 Oct 16 09:33 .
drwxr-xr-x 1 root root     4096 Oct 16 09:33 ..
-rw------- 1 root root     5609 Oct 16 09:33 eval_harness.py
-rw------- 1 root root 53544459 Oct 16 09:33 pairs.csv
-rw------- 1 root root    10097 Oct 16 09:33 sequences.csv
-rw------- 1 root root    10617 Oct 16 09:33 train_m1.py
-rw------- 1 root root 40442997 Oct 16 09:33 wt_emb_650.pt


In [ ]:
!python3 train_m1.py pairs.csv sequences.csv --esm-model esm2_t33_650M_UR50D \
    --cache wt_emb_650.pt --no-singles --out m1_ns_preds.csv

Loading cached WT embeddings from wt_emb_650.pt
Embedding dim = 1280; building features for 235355 pairs ...
no-singles mode: z_a/z_b hidden; target = fitted epistasis label
fold 0: trained on 188496, predicted 46859  (final train MSE 0.130)
fold 1: trained on 187989, predicted 47366  (final train MSE 0.138)
fold 2: trained on 188446, predicted 46909  (final train MSE 0.160)
fold 3: trained on 187954, predicted 47401  (final train MSE 0.118)
fold 4: trained on 188535, predicted 46820  (final train MSE 0.143)

Wrote m1_ns_preds.csv. Now score it:
  python3 eval_harness.py pairs.csv --pred m1_ns_preds.csv --name M1_ESM


In [ ]:
!python3 eval_harness.py pairs.csv --pred m1_ns_preds.csv --name M1_nosingles


=== M1_nosingles  (held-out-protein folds already baked into pairs.csv) ===
pairs evaluated: 235355   assays: 48
MACRO  (mean over assays): z_Spearman=0.739  z_RMSE=0.529  epi_Spearman=0.211  epi_AUROC=0.565
POOLED (all rows)        : z_Spearman=0.870  z_RMSE=0.554  epi_Spearman=0.095  epi_AUROC=0.565

By selection type (macro epi_Spearman):
  Activity             assays= 4  epi_Spearman=0.037
  Binding              assays= 4  epi_Spearman=0.095
  Expression           assays= 1  epi_Spearman=0.010
  OrganismalFitness    assays= 5  epi_Spearman=0.074
  Stability            assays=34  epi_Spearman=0.271

Per-assay metrics written to eval_per_assay.csv


In [ ]:
!python3 train_m1.py pairs.csv sequences.csv --esm-model esm2_t33_650M_UR50D \
    --cache wt_emb_650.pt --no-singles --save-final m1_nosingles_final.pt

Loading cached WT embeddings from wt_emb_650.pt
Embedding dim = 1280; building features for 235355 pairs ...
no-singles mode: z_a/z_b hidden; target = fitted epistasis label
Saved final M1 (trained on all 235,355 pairs) to m1_nosingles_final.pt (final train MSE 0.133)


In [ ]:
!cp m1_ns_preds.csv m1_nosingles_final.pt /content/drive/MyDrive/bs6204/
!ls -la /content/drive/MyDrive/bs6204/ | grep -E "m1_ns|nosingles"

-rw-------  1 root root   812270 Oct 16 09:42 m1_nosingles_final.pt
-rw-------  1 root root 13684020 Oct 16 09:42 m1_ns_preds.csv
